# GLiNER2 with Katip's recipe · is GLiNER a better base than BERTurk?

**What this does.** Fine-tunes `fastino/gliner2-privacy-filter-PII-multi` (Fastino, Apache-2.0, 307M) on exactly Katip Mask Base v2's
training data: Katip's 8,000 synthetic legal documents + 17,967 real Turkish Wiki NER sentences, same three types
(person, address, health condition), 3 epochs. The only difference from Base is the model, so KVKK-Bench can answer the question.

Off the shelf this model already scores 95.0 Overall on KVKK-Bench (Base: 96.1) without seeing Turkish legal text, so it is the
strongest non-Katip starting point. It is 11x Base's size and does not run in a browser, so it would be a computer-only option.

| | |
|---|---|
| Time | ~5 min smoke test, then ~45-90 min on a free T4 |
| Output | `katip-gliner2.zip` (the fine-tuned model, ~1.2 GB) and the training scores |
| Next | put the zip in Downloads; Katip scores it on KVKK-Bench as `gliner2-katip` (rules + model, like Base) |

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

In [ ]:
# 1 · Check the GPU, get the code. (Takes ~1-2 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q 'gliner2==2.1.0' huggingface_hub", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end (Colab: key icon > secret HF_TOKEN). Nothing is hard-coded.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

## 2 · Smoke test (3-5 min): 300 docs, 1 epoch. If this fails, send the error; don't start the full run.

In [ ]:
!cd ml/gliner2 && python train_gliner2.py --limit 300 --epochs 1 --out out_smoke 2>&1 | tail -25

## 3 · Full run (~45-90 min)

In [ ]:
!cd ml/gliner2 && python train_gliner2.py --out out_gliner2 --epochs 3 --bs 8 2>&1 | grep -v Warning | tail -40

In [ ]:
print(open("ml/gliner2/out_gliner2/results.json").read())

In [ ]:
# Save: a zip to download (left panel > Files) and, if you set HF_TOKEN and UPLOAD = True, a PRIVATE Hugging Face upload.
import shutil, os
z = shutil.make_archive("katip-gliner2", "zip", "ml/gliner2/out_gliner2", "model")
shutil.copy("ml/gliner2/out_gliner2/results.json", "katip-gliner2-results.json")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z); files.download("katip-gliner2-results.json")
except Exception:
    pass
UPLOAD = False
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-gliner2-experiment", private=True, exist_ok=True)
    api.upload_folder(folder_path="ml/gliner2/out_gliner2/model", repo_id="f1k/katip-gliner2-experiment", commit_message="trained in a notebook (katip-gliner2)")
    print("uploaded to https://huggingface.co/f1k/katip-gliner2-experiment (private)")

**Next:** drop `katip-gliner2.zip` in Downloads and say so in the Katip thread; it gets scored on KVKK-Bench against Base v2 (paired bootstrap).